# 02 — What Is Optimal Transport?

Build the central idea of the whole course by hand: moving a pile of mass into a new shape at the least possible cost.

**Prerequisites:** `00/01_environment_setup`.

**What you'll be able to do**
- Describe optimal transport as moving a pile of mass at least cost.
- Read its three ingredients: the two distributions, the cost matrix, and the transport plan.
- Compute an exact optimal plan and the Wasserstein distance with `POT`.
- Explain why the Wasserstein distance measures how far mass must travel.

This is the idea the entire course is built around — and you can grasp it today, with nothing more than a tiny one-dimensional example and your own eyes.

**How this notebook works:** a short idea, then we run it, then we read the figure together — and repeat. Every figure is explained, so nothing stays mysterious. Let's begin.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ot

from qot_course import viz
from qot_course.intro import check_environment

np.random.seed(42)        # reproducibility
viz.use_course_style()    # consistent, legible figures

print("Environment:")
for package, version in check_environment().items():
    print(f"  {package:22s} {version or 'MISSING'}")

## 1. The idea: moving a pile of sand

In 1781 Gaspard Monge asked a concrete question: you have a pile of sand and a hole to
fill, and moving a shovel costs effort proportional to the distance. **What is the
cheapest way to move all the sand into the hole?**

Replace "pile of sand" by a **distribution** (how much mass sits at each position) and
you have optimal transport. We use two distributions over a few positions: a `source`
(the sand we have) and a `target` (the shape we want).

In [ ]:
positions = np.arange(9)

source = np.array([0.0, 1.0, 3.0, 5.0, 2.0, 0.0, 0.0, 0.0, 0.0])
target = np.array([0.0, 0.0, 0.0, 1.0, 2.0, 4.0, 2.0, 1.0, 0.0])
source = source / source.sum()   # normalise to probability mass (total = 1)
target = target / target.sum()

print("source sums to", round(source.sum(), 6), "| target sums to", round(target.sum(), 6))

In [ ]:
fig = viz.plot_distributions(source, target)
plt.show()

**Read the figure.** Violet bars = the `source` (the mass we *have*); amber bars = the
`target` (the mass we *want*). Both sum to 1 — same total "sand", different shape. The
source sits on the left, the target on the right: intuitively we must move mass
*rightward*. Optimal transport finds the cheapest way to do exactly that.

## 2. The cost of moving mass

Moving mass isn't free: the further it goes, the more it costs. We encode this in a
**cost matrix** `C`, where `C[i, j]` is the cost of moving one unit from position `i`
to position `j`. The natural choice here is the squared distance, $c(i, j) = (i - j)^2$
— the cost that defines the **2-Wasserstein** distance.

In [ ]:
pos = positions.reshape(-1, 1)
cost = ot.dist(pos, pos, metric="sqeuclidean")   # C[i, j] = (i - j)^2

fig = viz.plot_cost_matrix(cost)
plt.show()

**Read the figure.** Each cell `(i, j)` is the price of moving a unit from source
position `i` to target position `j`. The **diagonal is dark (zero)** — staying put is
free — and the **corners are bright** — moving all the way across is expensive. Optimal
transport will try to keep mass close to the diagonal.

## 3. A transport plan — and its two constraints

A **transport plan** `P` is a table: `P[i, j]` = how much mass we move from `i` to `j`.
It must respect two **marginal constraints**:

- every **row** sums to the source: $\sum_j P[i,j] = \mathrm{source}[i]$ (move out exactly what is there),
- every **column** sums to the target: $\sum_i P[i,j] = \mathrm{target}[j]$ (fill each spot exactly).

The total cost of a plan is $\sum_{i,j} P[i,j]\, C[i,j]$. Let's first try a *naive* plan
— the independent coupling $P = \mathrm{source} \otimes \mathrm{target}$ — which respects
the constraints but ignores the cost.

In [ ]:
naive_plan = np.outer(source, target)

# The marginal constraints must hold (up to floating point):
print("row sums match source:", np.allclose(naive_plan.sum(axis=1), source))
print("col sums match target:", np.allclose(naive_plan.sum(axis=0), target))

naive_cost = float(np.sum(naive_plan * cost))
print(f"naive plan cost = {naive_cost:.4f}")

fig = viz.plot_transport_plan(naive_plan, title="A naive plan (independent coupling)")
plt.show()

**Read the figure.** The naive plan spreads mass everywhere — bright cells far from the
diagonal mean "a lot of mass moved a long way". It is a *valid* plan (the marginals
check out above) but wasteful: its cost is printed above. Can we do better?

## 4. The optimal plan (Kantorovich)

Kantorovich's insight: minimise the total cost over **all** plans satisfying the two
marginal constraints. This is a *linear program*, and `POT`'s `ot.emd` solves it
exactly.

In [ ]:
optimal_plan = ot.emd(source, target, cost)
optimal_cost = float(np.sum(optimal_plan * cost))

saving = 100.0 * (naive_cost - optimal_cost) / naive_cost
print(f"optimal cost = {optimal_cost:.4f}   (naive was {naive_cost:.4f})")
print(f"the optimal plan is {saving:.0f}% cheaper than the naive one")

fig = viz.plot_transport_plan(optimal_plan, title="Optimal plan (minimal cost)")
plt.show()

In [ ]:
fig = viz.plot_transport_arrows(source, target, optimal_plan)
plt.show()

**Read the two figures.** In the heatmap, the optimal plan hugs the **diagonal**: mass
moves only as far as it must. In the flow view, each green line carries mass from a
source dot (top) to a target dot (bottom), and its **width is proportional to the amount
moved** — you can literally see the sand sliding a few positions to the right. The
optimal cost is the smallest possible total effort (the saving vs. the naive plan is
printed above).

## 5. The Wasserstein distance

The minimal cost *is* the (squared) **2-Wasserstein distance** between the two
distributions:

$$ W_2(\mathrm{source}, \mathrm{target}) = \Big(\min_P \sum_{i,j} P[i,j]\,(i-j)^2\Big)^{1/2}. $$

Unlike comparing histograms bin-by-bin, $W_2$ knows about *geometry*: it measures how
far mass must travel. (Using $|i-j|$ instead of $(i-j)^2$ gives $W_1$, the
"earth-mover's distance".)

In [ ]:
w2 = float(np.sqrt(optimal_cost))
# ot.emd2 returns the optimal transport *cost* directly:
w2_check = float(np.sqrt(ot.emd2(source, target, cost)))
print(f"W2 = {w2:.4f}   (cross-check via ot.emd2: {w2_check:.4f})")

### Intuition: $W_2$ grows as the piles move apart

Keep the source fixed and slide the target further away: $W_2$ should grow steadily,
because the mass has to travel further. Let's check.

In [ ]:
def gaussian_bump(center, n=11, width=1.0):
    x = np.arange(n)
    b = np.exp(-0.5 * ((x - center) / width) ** 2)
    return b / b.sum()

n = 11
ref = gaussian_bump(3, n)
grid = np.arange(n).reshape(-1, 1)
shift_cost = ot.dist(grid, grid, metric="sqeuclidean")

shifts = np.arange(0, 7)
w2_curve = [np.sqrt(ot.emd2(ref, gaussian_bump(3 + d, n), shift_cost)) for d in shifts]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(shifts, w2_curve, "-o", color=viz.FLOW_COLOR, markersize=8)
ax.set_xlabel("how far the target bump is shifted (positions)")
ax.set_ylabel("$W_2$ distance")
ax.set_title("Wasserstein-2 grows with displacement", pad=12)
plt.show()

**Read the figure.** A clean, increasing curve: the more we displace the target, the
larger $W_2$. For two identical bumps a distance $d$ apart, $W_2 \approx d$ — the
Wasserstein distance is, quite literally, *how far you must move the mass*. This
geometric sensitivity is why optimal transport is so useful across data science.

## 6. The quantum twist — where we are heading

Everything above used **probability vectors**. The whole course is about what happens when we replace them by **density matrices** $\rho$ (the objects of quantum mechanics). We grow this dictionary as the course goes, one row at a time:

| Classical | Quantum |
|-----------|---------|
| probability vector `p` | density matrix $\rho$ (diagonal $\Rightarrow$ classical) |
| marginal | partial trace |
| transport plan | quantum coupling / channel |

The punchline you'll earn: a *diagonal* $\rho$ makes quantum optimal transport collapse back to today's classical optimal transport — so the genuinely new behaviour comes from the **off-diagonal** parts (coherences).

**The four movements of the course:** (1) quantum foundations → (2) information theory & geometry → (3) classical optimal transport → (4) quantum optimal transport.

## 7. Your turn

1. **Reshape the piles.** Change `source` and `target` in §1 to two new shapes, re-run,
   and watch where the optimal plan puts its mass. Does it still hug the diagonal?
2. **$W_1$ vs $W_2$.** Rebuild the cost with `metric="euclidean"` (so $c = |i-j|$) and
   compare the optimal plan and distance to the squared-cost case. Which splits mass more?
3. **Predict, then check.** Before running §5's curve, sketch what you expect $W_2$ vs
   shift to look like. Were you right?

## What you built

- You framed optimal transport as moving mass at least cost — defined by **two distributions + a cost matrix**, solved as a **transport plan**.
- You computed the **Wasserstein distance** and saw it measure geometric displacement, not a bin-by-bin difference.
- You watched $W_2$ grow as the piles move apart — a distance you can read straight off a figure.

You now hold the central idea of the course, built from scratch. Everything quantum we add later rests on exactly this picture — that is a real milestone for a first notebook.

## What's next

In module `01_QuantumFoundations` we start the quantum side from zero: what a qubit is, and the density matrix that will eventually take the place of the probability vector you used today.

## References

- G. Peyré & M. Cuturi, *Computational Optimal Transport*, Foundations and Trends in ML (2019). DOI:10.1561/2200000073
- C. Villani, *Optimal Transport: Old and New*, Springer (2009).

**Previous:** `notebooks/00_GettingStarted/01_environment_setup.ipynb` · **Next:** `notebooks/01_QuantumFoundations/01_amplitudes_and_superposition.ipynb`